<a href="https://colab.research.google.com/github/flalfud1024-dev/-/blob/claude%2Ffix-voice-recognition-cg5jV/web_scraping_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 웹 크롤링 실습: 독자 서평 데이터 수집

## 한강 『채식주의자』 한·중 독자 서평 수집 실습

---

### 수업 목표
1. 웹 크롤링의 기본 원리 이해
2. Python 라이브러리 (Requests, BeautifulSoup, Selenium) 활용법 습득
3. 더우반(豆瓣)과 Yes24에서 실제 독자 서평 데이터 수집
4. 수집한 데이터를 분석 가능한 형태로 정제

---
## Part 1: 웹 크롤링 기초 이론
---

### 1.1 웹 크롤링이란?

**웹 크롤링(Web Crawling)** 또는 **웹 스크래핑**(Web Scraping)은 웹 페이지에서 데이터를 자동으로 수집하는 기술입니다.

#### 크롤링의 기본 원리

```
┌─────────────┐     HTTP 요청      ┌─────────────┐
│   Python    │ ───────────────▶ │  웹 서버    │
│   스크립트  │                                  │  (Douban,   │
│             │ ◀─────────────── │   Yes24)    │
└─────────────┘     HTML 응답      └─────────────┘
       │
       ▼
┌─────────────┐
│  HTML 파싱   │
│  (분석/추출) │
└─────────────┘
       │
       ▼
┌─────────────┐
│ 데이터 저장  │
│ (CSV/Excel)  │
└─────────────┘
```

#### 주요 라이브러리

| 라이브러리 | 용도 | 특징 |
|-----------|------|------|
| **Requests** | HTTP 요청 | 간단하고 빠름, 정적 페이지에 적합 |
| **BeautifulSoup4** | HTML 파싱 | HTML 구조 분석 및 데이터 추출 |
| **Selenium** | 브라우저 자동화 | 동적 페이지(JavaScript) 처리 가능 |

### 1.2 정적 페이지 vs 동적 페이지

#### 정적 페이지 (Static Page)
- 서버에서 완성된 HTML을 바로 전송
- **Requests + BeautifulSoup**으로 충분
- 예: 더우반 독서 (豆瓣讀書)

#### 동적 페이지 (Dynamic Page)
- JavaScript로 콘텐츠를 동적으로 로드
- **Selenium** 필요 (실제 브라우저 구동)
- 예: Yes24 사락(Sarak) 커뮤니티

### 1.3 크롤링 시 주의사항 ⚠️

1. **robots.txt 확인**: 웹사이트의 크롤링 정책 준수
2. **요청 간격 조절**: 서버에 부담을 주지 않도록 딜레이 설정
3. **User-Agent 설정**: 적절한 헤더 정보 포함
4. **저작권 및 이용약관**: 수집 데이터의 사용 범위 확인
5. **학술 목적**: 연구 목적의 데이터 수집임을 명시

---
## Part 2: 환경 설정
---

In [ ]:
# 필요한 라이브러리 설치
!pip install requests beautifulsoup4 pandas lxml selenium webdriver-manager -q

print("✅ 라이브러리 설치 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 3.8 MB/s eta 0:00:00
✅ 라이브러리 설치 완료!


In [ ]:
# 라이브러리 임포트
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료!")
print(f"📅 현재 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ 라이브러리 임포트 완료!
📅 현재 시간: 2026-02-05 02:40:32


---
## Part 3: 크롤링 기초 실습
---

### 3.1 HTTP 요청의 이해

웹 페이지에 접근할 때 브라우저가 하는 일을 Python으로 재현합니다.

In [ ]:
# 간단한 HTTP 요청 예시
url = "https://httpbin.org/get"  # 테스트용 사이트

# GET 요청 보내기
response = requests.get(url)

print(f"📡 상태 코드: {response.status_code}")
print(f"📝 응답 타입: {response.headers.get('Content-Type')}")
print("\n--- 응답 내용 (일부) ---")
print(response.text[:500])

📡 상태 코드: 200
📝 응답 타입: application/json

--- 응답 내용 (일부) ---
{
  "args": {}, 
  "headers": {
    "Accept": "*/*", 
    "Accept-Encoding": "gzip, deflate, br", 
    "Host": "httpbin.org", 
    "User-Agent": "python-requests/2.32.4", 
    "X-Amzn-Trace-Id": "Root=1-69840399-2252aca105907acb0383ddb9"
  }, 
  "origin": "34.41.91.80", 
  "url": "https://httpbin.org/get"
}



### 3.2 HTTP 상태 코드

| 코드 | 의미 | 설명 |
|------|------|------|
| 200 | OK | 요청 성공 |
| 403 | Forbidden | 접근 거부 (User-Agent 필요) |
| 404 | Not Found | 페이지 없음 |
| 429 | Too Many Requests | 요청 과다 |
| 503 | Service Unavailable | 서버 과부하 |

In [ ]:
# User-Agent 헤더 설정의 중요성
# 많은 웹사이트가 봇 접근을 차단하므로 브라우저처럼 보이게 설정

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7,zh-CN;q=0.6,zh;q=0.5',
}

print("📋 설정된 헤더:")
for key, value in headers.items():
    print(f"  {key}: {value[:50]}..." if len(value) > 50 else f"  {key}: {value}")

📋 설정된 헤더:
  User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb...
  Accept: text/html,application/xhtml+xml,application/xml;q=...
  Accept-Language: ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7,zh-CN;q=0.6,zh...


### 3.3 HTML 파싱 기초

BeautifulSoup을 사용하여 HTML 구조를 분석하고 원하는 데이터를 추출합니다.

In [ ]:
# HTML 파싱 예시
sample_html = """
<html>
<head><title>예시 페이지</title></head>
<body>
    <div class="review-container">
        <div class="review" id="review1">
            <span class="user">김독자</span>
            <span class="rating">★★★★★</span>
            <p class="content">정말 감동적인 책이었습니다.</p>
            <span class="date">2024-01-15</span>
        </div>
        <div class="review" id="review2">
            <span class="user">이서평</span>
            <span class="rating">★★★★</span>
            <p class="content">문체가 아름다워요.</p>
            <span class="date">2024-01-16</span>
        </div>
    </div>
</body>
</html>
"""

# BeautifulSoup으로 파싱
soup = BeautifulSoup(sample_html, 'lxml')

print("📝 페이지 제목:", soup.title.string)
print("\n" + "="*50)
print("📖 리뷰 데이터 추출:")
print("="*50)

# 모든 리뷰 찾기
reviews = soup.find_all('div', class_='review')

for i, review in enumerate(reviews, 1):
    user = review.find('span', class_='user').text
    rating = review.find('span', class_='rating').text
    content = review.find('p', class_='content').text
    date = review.find('span', class_='date').text

    print(f"\n리뷰 {i}:")
    print(f"  👤 작성자: {user}")
    print(f"  ⭐ 평점: {rating}")
    print(f"  💬 내용: {content}")
    print(f"  📅 날짜: {date}")

📝 페이지 제목: 예시 페이지

📖 리뷰 데이터 추출:

리뷰 1:
  👤 작성자: 김독자
  ⭐ 평점: ★★★★★
  💬 내용: 정말 감동적인 책이었습니다.
  📅 날짜: 2024-01-15

리뷰 2:
  👤 작성자: 이서평
  ⭐ 평점: ★★★★
  💬 내용: 문체가 아름다워요.
  📅 날짜: 2024-01-16


### 3.4 BeautifulSoup 주요 메서드

| 메서드 | 설명 | 예시 |
|--------|------|------|
| `find()` | 첫 번째 일치 요소 | `soup.find('div', class_='review')` |
| `find_all()` | 모든 일치 요소 | `soup.find_all('span', class_='user')` |
| `select()` | CSS 선택자 사용 | `soup.select('div.review > p')` |
| `select_one()` | CSS 선택자 (첫 번째) | `soup.select_one('#review1')` |
| `.text` / `.get_text()` | 텍스트 추출 | `element.text` |
| `.get()` | 속성값 추출 | `element.get('href')` |

In [ ]:
# CSS 선택자 사용 예시
print("🎯 CSS 선택자 활용:")
print("\n1. 클래스로 선택: div.review")
for item in soup.select('div.review'):
    print(f"   - ID: {item.get('id')}")

print("\n2. ID로 선택: #review1")
review1 = soup.select_one('#review1')
print(f"   - 내용: {review1.find('p').text}")

print("\n3. 자식 요소 선택: div.review p.content")
contents = soup.select('div.review p.content')
for c in contents:
    print(f"   - {c.text}")

🎯 CSS 선택자 활용:

1. 클래스로 선택: div.review
   - ID: review1
   - ID: review2

2. ID로 선택: #review1
   - 내용: 정말 감동적인 책이었습니다.

3. 자식 요소 선택: div.review p.content
   - 정말 감동적인 책이었습니다.
   - 문체가 아름다워요.


---
## Part 4: 더우반(豆瓣) 단평 크롤링
---

더우반은 중국 최대의 도서/영화 리뷰 플랫폼입니다. 정적 페이지로 구성되어 있어 Requests + BeautifulSoup으로 크롤링이 가능합니다.

### 대상 URL
1. 素食者 (2021) - 후쟈오퉁 번역본
2. 素食者 (2016) - 다른 번역본  
3. 素食主义者 (2013) - 첸르 번역본

In [ ]:
# 더우반 크롤링 설정
class DoubanScraper:
    """
    더우반 도서 단평(短评) 크롤러
    """

    def __init__(self):
        self.session = requests.Session()
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,ko;q=0.7',
            'Referer': 'https://book.douban.com/',
        }
        self.session.headers.update(self.headers)

    def get_page(self, url, retry=3):
        """
        페이지 요청 (재시도 로직 포함)
        """
        for attempt in range(retry):
            try:
                response = self.session.get(url, timeout=10)
                if response.status_code == 200:
                    return response
                elif response.status_code == 403:
                    print(f"⚠️ 접근 거부 (403). 잠시 대기 후 재시도...")
                    time.sleep(random.uniform(5, 10))
                else:
                    print(f"⚠️ 상태 코드: {response.status_code}")
            except Exception as e:
                print(f"❌ 오류 발생: {e}")
                time.sleep(random.uniform(3, 5))
        return None

    def parse_comments(self, html_content):
        """
        단평 파싱
        """
        soup = BeautifulSoup(html_content, 'lxml')
        comments = []

        # 단평 컨테이너 찾기
        comment_items = soup.select('li.comment-item')

        for item in comment_items:
            try:
                # 사용자명
                user_elem = item.select_one('span.comment-info a')
                username = user_elem.text.strip() if user_elem else 'Unknown'

                # 평점 (별 개수)
                rating_elem = item.select_one('span.comment-info span[class*="rating"]')
                if rating_elem:
                    rating_class = rating_elem.get('class', [])
                    rating = next((c for c in rating_class if 'allstar' in c), None)
                    rating = int(rating.replace('allstar', '')) // 10 if rating else 0
                else:
                    rating = 0

                # 날짜
                date_elem = item.select_one('span.comment-info span.comment-time')
                date = date_elem.text.strip() if date_elem else ''

                # 단평 내용
                content_elem = item.select_one('span.short')
                content = content_elem.text.strip() if content_elem else ''

                # 유용함 투표
                vote_elem = item.select_one('span.vote-count')
                votes = int(vote_elem.text) if vote_elem and vote_elem.text.isdigit() else 0

                if content:  # 내용이 있는 경우만 추가
                    comments.append({
                        'username': username,
                        'rating': rating,
                        'date': date,
                        'content': content,
                        'votes': votes
                    })
            except Exception as e:
                print(f"⚠️ 파싱 오류: {e}")
                continue

        return comments

    def scrape_book_comments(self, book_id, max_pages=5, status='P'):
        """
        특정 도서의 단평 수집

        Parameters:
        - book_id: 도서 ID
        - max_pages: 최대 페이지 수
        - status: 'P' (읽음), 'F' (읽는 중), 'W' (읽고 싶음)
        """
        all_comments = []
        base_url = f"https://book.douban.com/subject/{book_id}/comments/"

        print(f"📚 도서 ID {book_id} 단평 수집 시작...")
        print(f"📄 최대 {max_pages} 페이지 수집 예정")
        print("="*50)

        for page in range(max_pages):
            start = page * 20
            url = f"{base_url}?start={start}&limit=20&status={status}&sort=score"

            print(f"\n📖 페이지 {page + 1}/{max_pages} 수집 중... (start={start})")

            response = self.get_page(url)
            if not response:
                print(f"❌ 페이지 {page + 1} 수집 실패")
                continue

            comments = self.parse_comments(response.text)

            if not comments:
                print(f"⚠️ 더 이상 단평이 없습니다.")
                break

            all_comments.extend(comments)
            print(f"   ✅ {len(comments)}개 단평 수집 완료 (총 {len(all_comments)}개)")

            # 서버 부하 방지를 위한 대기
            delay = random.uniform(2, 4)
            print(f"   ⏳ {delay:.1f}초 대기...")
            time.sleep(delay)

        print("\n" + "="*50)
        print(f"🎉 수집 완료! 총 {len(all_comments)}개의 단평")

        return pd.DataFrame(all_comments)

print("✅ DoubanScraper 클래스 정의 완료!")

✅ DoubanScraper 클래스 정의 완료!


### 4.1 더우반 단평 수집 실습

⚠️ **주의**: 더우반은 IP 차단이 있을 수 있으므로 요청 간격을 충분히 두어야 합니다.

In [ ]:
# 크롤러 인스턴스 생성
douban_scraper = DoubanScraper()

# 대상 도서 정보
books = {
    '35534519': '素食者 (2021) - 후쟈오퉁 역',
    '26735623': '素食者 (2016)',
    '24847418': '素食主义者 (2013) - 첸르 역'
}

print("📚 수집 대상 도서:")
for book_id, title in books.items():
    print(f"  • {title} (ID: {book_id})")

📚 수집 대상 도서:
  • 素食者 (2021) - 후쟈오퉁 역 (ID: 35534519)
  • 素食者 (2016) (ID: 26735623)
  • 素食主义者 (2013) - 첸르 역 (ID: 24847418)


In [ ]:
# 실습: 2021년 번역본 단평 수집 (3페이지만)
# 실제 연구에서는 더 많은 페이지를 수집합니다.

print("🚀 더우반 단평 크롤링 시작!")
print("\n⚠️ 참고: 더우반 서버 상태에 따라 일부 요청이 실패할 수 있습니다.")
print("   Colab 환경에서는 IP 제한이 있을 수 있으니 소량만 테스트합니다.\n")

# 2021년 번역본 (후쟈오퉁 역) - 테스트로 2페이지만
df_douban_2021 = douban_scraper.scrape_book_comments('35534519', max_pages=10)

🚀 더우반 단평 크롤링 시작!

⚠️ 참고: 더우반 서버 상태에 따라 일부 요청이 실패할 수 있습니다.
   Colab 환경에서는 IP 제한이 있을 수 있으니 소량만 테스트합니다.

📚 도서 ID 35534519 단평 수집 시작...
📄 최대 10 페이지 수집 예정

📖 페이지 1/10 수집 중... (start=0)
   ✅ 20개 단평 수집 완료 (총 20개)
   ⏳ 3.5초 대기...

📖 페이지 2/10 수집 중... (start=20)
   ✅ 20개 단평 수집 완료 (총 40개)
   ⏳ 2.5초 대기...

📖 페이지 3/10 수집 중... (start=40)
   ✅ 20개 단평 수집 완료 (총 60개)
   ⏳ 3.2초 대기...

📖 페이지 4/10 수집 중... (start=60)
   ✅ 20개 단평 수집 완료 (총 80개)
   ⏳ 2.5초 대기...

📖 페이지 5/10 수집 중... (start=80)
   ✅ 20개 단평 수집 완료 (총 100개)
   ⏳ 2.2초 대기...

📖 페이지 6/10 수집 중... (start=100)
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
❌ 페이지 6 수집 실패

📖 페이지 7/10 수집 중... (start=120)
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
❌ 페이지 7 수집 실패

📖 페이지 8/10 수집 중... (start=140)
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
❌ 페이지 8 수집 실패

📖 페이지 9/10 수집 중... (start=160)
⚠️ 접근 거부 (403). 잠시 대기 후 재시도...
⚠️ 접근 

In [ ]:
# 수집 결과 확인
if len(df_douban_2021) > 0:
    print("📊 수집된 데이터 미리보기:")
    print("="*80)
    display(df_douban_2021.head(10))

    print("\n📈 기본 통계:")
    print(f"  • 총 단평 수: {len(df_douban_2021)}")
    print(f"  • 평균 평점: {df_douban_2021['rating'].mean():.2f}")
    print(f"  • 평점 분포:")
    print(df_douban_2021['rating'].value_counts().sort_index())
else:
    print("⚠️ 수집된 데이터가 없습니다. 더우반 접근이 제한되었을 수 있습니다.")
    print("   이 경우 아래의 샘플 데이터를 사용하여 진행합니다.")

In [ ]:
# 샘플 데이터 (접근 제한 시 사용)
sample_douban_data = [
    {'username': '李**', 'rating': 5, 'date': '2021-09-12',
     'content': '过于令人惊艳的作品。读的过程中时刻联想到《狂人日记》。被视为"受害狂"的狂人看穿了本质伪善的社会中比比皆是"吃人者"',
     'votes': 156},
    {'username': '麦**miki', 'rating': 4, 'date': '2022-08-21',
     'content': '看得越多，越会对东亚文化体系下的暴力有着更为深刻的理解和认同。韩江跟金爱烂都同属于擅长描写主角细腻内心的那一类型',
     'votes': 89},
    {'username': '楹*', 'rating': 2, 'date': '2024-10-14',
     'content': '读着顺畅但毫无感触。写作最难的是驾驭荒谬感。莫名且有意堆砌的荒谬则会让人感到无聊',
     'votes': 45},
    {'username': '桃****生', 'rating': 4, 'date': '2021-09-09',
     'content': '之前读过不少韩国小说都是社会流，这给我一种感觉就是韩国人很擅于剖析社会。但是不擅长写"真正的文学"',
     'votes': 78},
    {'username': '未***柔', 'rating': 3, 'date': '2021-09-24',
     'content': '不知道是不是翻译的问题，很多描述让人觉得又急又浅',
     'votes': 112},
]

if len(df_douban_2021) == 0:
    df_douban_2021 = pd.DataFrame(sample_douban_data)
    print("📋 샘플 데이터 로드 완료!")
    display(df_douban_2021)

### 4.2 더우반 장평(書評) 크롤링

장평은 단평보다 길고 상세한 리뷰입니다.

In [ ]:
def scrape_douban_reviews(book_id, max_pages=2):
    """
    더우반 장평(書評) 크롤링
    """
    session = requests.Session()
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept-Language': 'zh-CN,zh;q=0.9',
    }

    all_reviews = []
    base_url = f"https://book.douban.com/subject/{book_id}/reviews"

    print(f"📝 장평 수집 시작 (도서 ID: {book_id})")

    for page in range(max_pages):
        start = page * 20
        url = f"{base_url}?start={start}"

        try:
            response = session.get(url, headers=headers, timeout=10)
            if response.status_code != 200:
                print(f"⚠️ 페이지 {page+1} 접근 실패: {response.status_code}")
                continue

            soup = BeautifulSoup(response.text, 'lxml')
            review_items = soup.select('div.review-item')

            if not review_items:
                print(f"⚠️ 더 이상 장평이 없습니다.")
                break

            for item in review_items:
                try:
                    # 제목
                    title_elem = item.select_one('h2 a')
                    title = title_elem.text.strip() if title_elem else ''

                    # 작성자
                    author_elem = item.select_one('header.main-hd a.name')
                    author = author_elem.text.strip() if author_elem else ''

                    # 평점
                    rating_elem = item.select_one('span[class*="rating"]')
                    rating = 0
                    if rating_elem:
                        rating_class = rating_elem.get('class', [])
                        for c in rating_class:
                            if 'allstar' in c:
                                rating = int(c.replace('allstar', '')) // 10
                                break

                    # 요약 내용
                    content_elem = item.select_one('div.short-content')
                    content = content_elem.text.strip() if content_elem else ''
                    content = re.sub(r'\s+', ' ', content)  # 공백 정리

                    if title and content:
                        all_reviews.append({
                            'title': title,
                            'author': author,
                            'rating': rating,
                            'content': content[:500]  # 500자로 제한
                        })
                except Exception as e:
                    continue

            print(f"  ✅ 페이지 {page+1}: {len(review_items)}개 장평 처리")
            time.sleep(random.uniform(2, 4))

        except Exception as e:
            print(f"❌ 오류: {e}")
            continue

    print(f"\n🎉 총 {len(all_reviews)}개 장평 수집 완료!")
    return pd.DataFrame(all_reviews)

print("✅ 장평 크롤링 함수 정의 완료!")

In [ ]:
# 장평 수집 테스트 (1페이지만)
df_reviews = scrape_douban_reviews('35534519', max_pages=1)

if len(df_reviews) > 0:
    display(df_reviews.head())
else:
    print("⚠️ 접근 제한으로 수집 실패. 샘플 데이터 사용.")

---
## Part 5: Yes24 크롤링 (Selenium 활용)
---

Yes24의 '사락(Sarak)' 커뮤니티는 JavaScript로 동적으로 로드되므로 Selenium이 필요합니다.

### Selenium 특징
- 실제 브라우저를 자동으로 제어
- JavaScript 렌더링 대기 가능
- 클릭, 스크롤 등 사용자 행동 시뮬레이션

In [ ]:
# Selenium 설정 (Colab 환경)
# Chrome 드라이버 자동 설치

try:
    from selenium import webdriver
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    from webdriver_manager.chrome import ChromeDriverManager

    print("✅ Selenium 임포트 완료!")
except ImportError as e:
    print(f"❌ Selenium 임포트 실패: {e}")
    print("   !pip install selenium webdriver-manager 실행 후 다시 시도하세요.")

In [ ]:
# Colab용 Chrome 설정
def setup_chrome_driver():
    """
    Colab 환경에서 Chrome 드라이버 설정
    """
    # Chrome 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # 헤드리스 모드 (화면 없이 실행)
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

    try:
        # 드라이버 자동 설치 및 초기화
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        print("✅ Chrome 드라이버 설정 완료!")
        return driver
    except Exception as e:
        print(f"❌ 드라이버 설정 실패: {e}")
        return None

print("✅ Chrome 드라이버 설정 함수 정의 완료!")

In [ ]:
class Yes24Scraper:
    """
    Yes24 리뷰/한줄평 크롤러 (Selenium 사용)
    """

    def __init__(self):
        self.driver = None

    def start_driver(self):
        """드라이버 시작"""
        if self.driver is None:
            self.driver = setup_chrome_driver()
        return self.driver is not None

    def close_driver(self):
        """드라이버 종료"""
        if self.driver:
            self.driver.quit()
            self.driver = None
            print("✅ 드라이버 종료")

    def scrape_reviews(self, product_id, max_pages=3):
        """
        Yes24 리뷰 수집
        """
        if not self.start_driver():
            print("❌ 드라이버 시작 실패")
            return pd.DataFrame()

        all_reviews = []
        base_url = f"https://www.yes24.com/Product/communityMod498"

        print(f"📚 Yes24 리뷰 수집 시작 (상품 ID: {product_id})")

        try:
            # 상품 페이지 접근
            url = f"https://www.yes24.com/Product/Goods/{product_id}"
            self.driver.get(url)
            time.sleep(3)

            print(f"📄 페이지 로드 완료: {self.driver.title}")

            # 리뷰 탭 클릭 시도
            try:
                review_tab = WebDriverWait(self.driver, 10).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "a[href*='#review']"))
                )
                review_tab.click()
                time.sleep(2)
            except:
                print("⚠️ 리뷰 탭을 찾을 수 없습니다.")

            # 리뷰 추출
            soup = BeautifulSoup(self.driver.page_source, 'lxml')

            # 리뷰 항목 찾기 (Yes24 HTML 구조에 맞게 조정 필요)
            review_items = soup.select('div.review_cont, div.reviewInfoBot, li.reviewInfoItem')

            for item in review_items:
                try:
                    content = item.get_text(strip=True)
                    if len(content) > 10:  # 최소 길이 필터
                        all_reviews.append({
                            'content': content[:500],
                            'source': 'yes24'
                        })
                except:
                    continue

            print(f"✅ {len(all_reviews)}개 리뷰 수집")

        except Exception as e:
            print(f"❌ 오류 발생: {e}")

        return pd.DataFrame(all_reviews)

    def scrape_with_requests(self, product_id):
        """
        Requests로 기본 정보 수집 (Selenium 없이)
        """
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept-Language': 'ko-KR,ko;q=0.9',
        }

        url = f"https://www.yes24.com/Product/Goods/{product_id}"

        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'lxml')

                # 책 제목
                title = soup.select_one('h2.gd_name')
                title = title.text.strip() if title else 'Unknown'

                # 평점
                rating = soup.select_one('span.gd_rating em')
                rating = rating.text.strip() if rating else 'N/A'

                print(f"📚 책 제목: {title}")
                print(f"⭐ 평점: {rating}")

                return {'title': title, 'rating': rating}
        except Exception as e:
            print(f"❌ 오류: {e}")

        return None

print("✅ Yes24Scraper 클래스 정의 완료!")

In [ ]:
# Yes24 기본 정보 수집 테스트 (Requests 사용)
yes24_scraper = Yes24Scraper()

# 채식주의자 상품 ID
product_id = "108422348"

print("📖 Yes24 도서 정보 조회...")
book_info = yes24_scraper.scrape_with_requests(product_id)

In [ ]:
# Selenium을 사용한 리뷰 수집 (선택적 실행)
# ⚠️ Colab 환경에서는 실행 시간이 오래 걸릴 수 있습니다.

RUN_SELENIUM = False  # True로 변경하면 Selenium 실행

if RUN_SELENIUM:
    print("🚀 Selenium으로 리뷰 수집 시작...")
    df_yes24 = yes24_scraper.scrape_reviews(product_id, max_pages=1)
    yes24_scraper.close_driver()

    if len(df_yes24) > 0:
        display(df_yes24.head())
else:
    print("ℹ️ Selenium 실행이 비활성화되어 있습니다.")
    print("   RUN_SELENIUM = True로 변경하면 실행됩니다.")

### 5.1 Yes24 샘플 데이터

논문에서 인용된 한국 독자 서평 예시입니다.

In [ ]:
# 논문에서 인용된 한국 독자 서평 샘플
sample_yes24_data = [
    {
        'username': 'j***125',
        'date': '2025-01-27',
        'content': '우선 노벨문학상 수상을 축하합니다. 작가의 세밀한 감성으로 그려낸 작품, 똑같은 사물이나 현상에 대해서 느끼는 감정이나 사고 과정은 당연히 다르겠지만 작가의 그것은 한 차원 높은 세밀한 시각과 표현일 것이다.',
        'rating': 5
    },
    {
        'username': 'H****er',
        'date': '2024-12-01',
        'content': '작가가 사용하는 단어들은 너무나도 찬란하다. 노벨 문학상 수상 작가의 책을 원서로 읽고 그 원서 특히 내 모국어로 써 있는 경우에 그것은 매우 특별한 것이다. 가독성은 번역에 따지는 것이지만, 한강의 문체는 마치 소설 전체가 시처럼 느껴졌다.',
        'rating': 5
    },
    {
        'username': '정*리',
        'date': '2024-11-15',
        'content': '채식주의자를 처음 1회독했을 때는 불쾌했고, 2회독했을 때는 슬펐고, 3회독했을 때는 감탄했다. 처음엔 생생한 이미지가, 두 번째는 인간의 가련함에, 세 번째는 섬세한 직조에... 한강의 문학은 곱씹을수록 찬란하다.',
        'rating': 5
    },
    {
        'username': 'k****h',
        'date': '2024-12-15',
        'content': '소설에 재미를 느끼게 해 준 책입니다. 처음엔 노벨 수상자의 책이라 읽어볼까 하고 시작한 건데 너무 재미있게 읽었습니다. 같은 내용을 저는 저런 문장으로 표현할 수 있었을까에 대해서도 생각했는데 문장이 너무 아름답고 정말 대단하다는 말밖에 할 수 없습니다.',
        'rating': 5
    },
    {
        'username': 'j*****04',
        'date': '2025-04-17',
        'content': '지독하게 어렵다. 그럼에도 처절하게 이끌린다. 불쾌함이 덩어리진 이야기, 그 속에서 살아가는 영혜를 보며 강요되고 억압되었던 이 사회의 문제점들을 비로소 돌아보게 된다.',
        'rating': 4
    },
    {
        'username': '지*군',
        'date': '2024-12-03',
        'content': '읽으면 읽을수록 충격과 가슴속에서 무언가 걸린 듯한 답답함의 무저갱으로 끌려가는 느낌이었다. 챕터 하나하나에 감정 소모가 굉장해서 한동안 다신 읽고 싶진 않았다. 그러나 문체와 필력이 갖고 오는 막중한 무게의 답답함은 그 어느 책, 매체보다 강렬했다.',
        'rating': 4
    },
]

df_yes24_sample = pd.DataFrame(sample_yes24_data)
print("📋 한국 독자 서평 샘플 데이터:")
display(df_yes24_sample)

---
## Part 6: 데이터 정제 및 저장
---

In [ ]:
# 데이터 정제 함수
def clean_text(text):
    """
    텍스트 정제
    """
    if not isinstance(text, str):
        return ''

    # 공백 정규화
    text = re.sub(r'\s+', ' ', text)
    # 앞뒤 공백 제거
    text = text.strip()
    # 특수문자 일부 제거 (선택적)
    # text = re.sub(r'[\n\r\t]', ' ', text)

    return text

def process_dataframe(df, source='unknown'):
    """
    데이터프레임 처리
    """
    df = df.copy()

    # 내용 정제
    if 'content' in df.columns:
        df['content'] = df['content'].apply(clean_text)
        df['content_length'] = df['content'].apply(len)

    # 소스 표시
    df['source'] = source

    # 빈 내용 제거
    if 'content' in df.columns:
        df = df[df['content'].str.len() > 5]

    return df

print("✅ 데이터 정제 함수 정의 완료!")

In [ ]:
# 데이터 처리
df_douban_processed = process_dataframe(df_douban_2021, source='douban')
df_yes24_processed = process_dataframe(df_yes24_sample, source='yes24')

print("📊 처리된 더우반 데이터:")
display(df_douban_processed.head())

print("\n📊 처리된 Yes24 데이터:")
display(df_yes24_processed.head())

In [ ]:
# CSV 파일로 저장
from google.colab import files

# 저장
df_douban_processed.to_csv('douban_reviews.csv', index=False, encoding='utf-8-sig')
df_yes24_processed.to_csv('yes24_reviews.csv', index=False, encoding='utf-8-sig')

print("✅ CSV 파일 저장 완료!")
print("  📁 douban_reviews.csv")
print("  📁 yes24_reviews.csv")

In [ ]:
# 파일 다운로드 (Colab 환경)
try:
    files.download('douban_reviews.csv')
    files.download('yes24_reviews.csv')
    print("📥 파일 다운로드 시작!")
except:
    print("ℹ️ 파일은 Colab 파일 탭에서 다운로드할 수 있습니다.")

---
## Part 7: 기본 분석 미리보기
---

In [ ]:
# 기본 통계 비교
print("="*60)
print("📊 한중 독자 서평 기본 통계 비교")
print("="*60)

print("\n🇨🇳 중국 독자 (더우반):")
print(f"  • 총 서평 수: {len(df_douban_processed)}")
if 'rating' in df_douban_processed.columns:
    print(f"  • 평균 평점: {df_douban_processed['rating'].mean():.2f}")
if 'content_length' in df_douban_processed.columns:
    print(f"  • 평균 글자 수: {df_douban_processed['content_length'].mean():.1f}")

print("\n🇰🇷 한국 독자 (Yes24):")
print(f"  • 총 서평 수: {len(df_yes24_processed)}")
if 'rating' in df_yes24_processed.columns:
    print(f"  • 평균 평점: {df_yes24_processed['rating'].mean():.2f}")
if 'content_length' in df_yes24_processed.columns:
    print(f"  • 평균 글자 수: {df_yes24_processed['content_length'].mean():.1f}")

In [ ]:
# 간단한 키워드 분석
from collections import Counter

def simple_keyword_analysis(texts, language='ko'):
    """
    간단한 키워드 빈도 분석
    """
    # 모든 텍스트 합치기
    all_text = ' '.join(texts)

    if language == 'ko':
        # 한국어: 2글자 이상 단어 추출 (간단 버전)
        words = re.findall(r'[가-힣]{2,}', all_text)
    else:
        # 중국어: 2글자 이상 추출
        words = re.findall(r'[\u4e00-\u9fff]{2,}', all_text)

    # 빈도 계산
    word_freq = Counter(words)

    return word_freq.most_common(20)

print("📝 중국어 서평 주요 키워드 (단순 추출):")
cn_keywords = simple_keyword_analysis(df_douban_processed['content'].tolist(), 'zh')
for word, freq in cn_keywords[:10]:
    print(f"  {word}: {freq}")

print("\n📝 한국어 서평 주요 키워드 (단순 추출):")
kr_keywords = simple_keyword_analysis(df_yes24_processed['content'].tolist(), 'ko')
for word, freq in kr_keywords[:10]:
    print(f"  {word}: {freq}")

---
## 부록: 유용한 팁
---

### A. 크롤링 시 IP 차단 대응

```python
# 1. 요청 간격 늘리기
time.sleep(random.uniform(3, 7))

# 2. 여러 User-Agent 랜덤 사용
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36...',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36...',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36...'
]
headers['User-Agent'] = random.choice(user_agents)

# 3. 프록시 사용 (고급)
proxies = {'http': 'http://proxy:port', 'https': 'http://proxy:port'}
```

### B. robots.txt 확인

```python
# 웹사이트의 크롤링 정책 확인
robots_url = "https://book.douban.com/robots.txt"
response = requests.get(robots_url)
print(response.text)
```

### C. 에러 처리 패턴

```python
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # HTTP 에러 발생 시 예외
except requests.exceptions.Timeout:
    print("시간 초과")
except requests.exceptions.HTTPError as e:
    print(f"HTTP 에러: {e}")
except requests.exceptions.RequestException as e:
    print(f"요청 실패: {e}")
```

---
## 📚 참고 자료
---

### 논문
- 백선 (2025). 「문체 번역 가능성 연구: 『채식주의자』 한·중 독자 서평의 NLP 비교·분석」

### 라이브러리 문서
- [Requests 공식 문서](https://docs.python-requests.org/)
- [BeautifulSoup 문서](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Selenium Python 바인딩](https://selenium-python.readthedocs.io/)

### 데이터 소스
- [豆瓣讀書](https://book.douban.com/) - 중국 도서 리뷰
- [Yes24](https://www.yes24.com/) - 한국 온라인 서점